# Session 18: Confidence Intervals and Hypothesis Tests

Data Science with Python, Level 1, Stage 3: Statistics and ML Fundamentals

## Learning Objectives

By the end of this session you will be able to:

1. Explain why sample results vary.
2. Calculate a standard error.
3. Build confidence intervals for a mean and a proportion.
4. State a null and an alternative hypothesis.
5. Run a permutation test, a t-test and a chi-square test.
6. Interpret p-values and the two kinds of error.

## Where We Are

In Session 17 you saw that the means of many samples of dice rolls form a bell curve. That idea answers two of the most common questions in data work. **How sure are we?** A confidence interval gives a range of plausible values for a number we only estimated from a sample.

**Is the difference real?** A company shows half of its visitors a new page and half the old one, and the new page converts a little better. A hypothesis test tells us whether such a difference is larger than chance alone would produce. Today you answer both questions with simulation first, then with SciPy.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(28)
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
exams = pd.read_csv(DATA_URL + "exam_scores.csv")

rng = np.random.default_rng(30)
ab = pd.read_csv(DATA_URL + "ab_test.csv")

## 1. Samples Vary

To see sampling in action we invent a **population** we fully know: the commute times, in minutes, of 50,000 workers in a city. They are right-skewed, like many real measurements. Then we take samples and compare their means with the true mean.

In [ ]:
population = rng.gamma(shape=2.0, scale=15.0, size=50_000)
print("True mean:", round(population.mean(), 2))

for i in range(5):
    sample = rng.choice(population, size=40, replace=False)
    print(f"Sample {i + 1} mean: {sample.mean():.2f}")

## 2. The Sampling Distribution

In [ ]:
means_40 = [rng.choice(population, 40).mean() for _ in range(5000)]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sns.histplot(population, bins=50, ax=axes[0])
axes[0].set_title("Population: skewed")
sns.histplot(means_40, bins=50, ax=axes[1], color="tab:purple")
axes[1].set_title("Means of 5,000 samples of 40: a bell")
plt.show()

## 3. Confidence Intervals for a Mean

Because sample means are bell-shaped, about 95 percent of them fall within about 1.96 standard errors of the true mean. Turning this around: if we take the sample mean and go 1.96 standard errors either side, the interval catches the true mean in about 95 percent of samples. That interval is a **95 percent confidence interval**.

For small samples we use a slightly wider multiplier from the **t distribution** instead of 1.96. `stats.t.interval` does it for us.

In [ ]:
math = exams["math"]
n, mean = len(math), math.mean()
se = math.std() / np.sqrt(n)

print(f"Mean {mean:.2f}, standard error {se:.2f}")
print(f"Approximate 95% CI: {mean - 1.96 * se:.2f} to {mean + 1.96 * se:.2f}")
low, high = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)
print(f"t-based 95% CI:     {low:.2f} to {high:.2f}")

In [ ]:
hits = 0
for _ in range(1000):
    s = rng.choice(population, 40)
    se = s.std(ddof=1) / np.sqrt(40)
    low, high = stats.t.interval(0.95, df=39, loc=s.mean(), scale=se)
    hits += low <= population.mean() <= high

print("Intervals that caught the true mean:", hits / 1000)

## 4. Confidence Intervals for a Proportion

In [ ]:
p_hat = exams["passed"].mean()
n = len(exams)
se = np.sqrt(p_hat * (1 - p_hat) / n)

print(f"Pass rate {p_hat:.3f}, margin of error {1.96 * se:.3f}")
print(f"95% CI: {p_hat - 1.96 * se:.3f} to {p_hat + 1.96 * se:.3f}")

The second half uses random numbers. Resetting the generator here makes your results match the slides.

In [ ]:
rng = np.random.default_rng(30)

## 5. The Logic of a Test

In [ ]:
summary = ab.groupby("group").agg(visitors=("visitor_id", "count"),
                                  conversion=("converted", "mean"),
                                  time_on_page=("time_on_page_s", "mean"))
summary.round(3)

## 6. A Permutation Test

If the page made **no** difference (H0), the labels A and B are meaningless: any visitor could have been in either group. So we **shuffle** the labels many times, and each time compute the difference in conversion. This shows the differences that chance alone produces. The p-value is the share of shuffles with a difference at least as big as the real one.

In [ ]:
converted = ab["converted"].to_numpy()
is_b = (ab["group"] == "B").to_numpy()
observed = converted[is_b].mean() - converted[~is_b].mean()

diffs = []
for _ in range(5000):
    shuffled = rng.permutation(is_b)
    diffs.append(converted[shuffled].mean() - converted[~shuffled].mean())
diffs = np.array(diffs)

print("Observed difference:", round(observed, 4))
print("p-value:", (np.abs(diffs) >= abs(observed)).mean())

In [ ]:
sns.histplot(diffs, bins=40)
plt.axvline(observed, color="tab:red", linewidth=2, label="observed")
plt.title("Differences produced by chance alone")
plt.xlabel("Difference in conversion rate")
plt.legend()
plt.show()

## 7. Tests in SciPy

In [ ]:
a = ab.loc[ab["group"] == "A", "time_on_page_s"]
b = ab.loc[ab["group"] == "B", "time_on_page_s"]

t, p = stats.ttest_ind(b, a, equal_var=False)
print(f"Mean A {a.mean():.1f} s, mean B {b.mean():.1f} s")
print(f"t = {t:.2f}, p = {p:.4f}")

In [ ]:
table = pd.crosstab(ab["group"], ab["converted"])
print(table)

chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"chi2 = {chi2:.1f}, p = {p:.2e}")

| Question | Data | SciPy |
| --- | --- | --- |
| Do two groups differ in their mean? | Numbers, two groups | `stats.ttest_ind(a, b, equal_var=False)` |
| Did the same people change? | Numbers, before and after | `stats.ttest_rel(before, after)` |
| Do several groups differ in their mean? | Numbers, three or more groups | `stats.f_oneway(a, b, c)` |
| Are two categories related? | Counts in a table | `stats.chi2_contingency(table)` |
| Any statistic, few assumptions | Anything | A permutation test |

## 8. Errors and Significance

In [ ]:
diff = b.mean() - a.mean()
pooled_sd = np.sqrt((a.var() + b.var()) / 2)
print(f"Difference: {diff:.1f} seconds")
print(f"Effect size (Cohen's d): {diff / pooled_sd:.2f}")

print("Conversion lift:", round((summary.loc["B", "conversion"] / summary.loc["A", "conversion"] - 1) * 100, 1), "percent")

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Wrong claim | Reading one interval as a 95 percent chance | Say: 95 percent of such intervals contain the true value. |
| Biased sample | Using a large but non-random sample | No formula fixes bias; only random sampling does. |
| Misread p | Saying p is the chance that H0 is true | p is the chance of data this extreme if H0 were true. |
| No difference proven | Concluding no effect because p > 0.05 | Say the data does not show an effect; it may be too small. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: English Interval

Build a 95 percent confidence interval for the mean English score with `stats.t.interval`.

In [ ]:
# Write your code below

### In-Class Exercise 2: Churn Rate

Load `telecom_churn.csv` and build a 95 percent confidence interval for the churn rate of customers on monthly contracts.

In [ ]:
# Write your code below

### In-Class Exercise 3: Devices

Does conversion differ between mobile and desktop visitors? Build the crosstab and run a chi-square test.

In [ ]:
# Write your code below

### In-Class Exercise 4: Sample Size Matters

Take only the first 200 visitors of the dataset and run the chi-square test on group and conversion again. Is the result still significant? Why might it change?

In [ ]:
# Write your code below

## Summary

- Sample means vary; their spread is the standard error, std divided by root n.
- A 95 percent interval is the estimate plus or minus about two standard errors.
- Bigger samples give narrower intervals.
- A test starts from H0, no difference, and asks how surprising the data is.
- A small p-value means the result is unlikely under H0.
- Significant is not the same as important; no difference is never proven.

## What's Next

Next session you will measure how strongly two variables move together, and build your first machine learning model: a line that predicts apartment prices from their features.